# ఖర్చు క్లెయిమ్ విశ్లేషణ

ఈ నోట్‌బుక్ లోకల్ రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను ప్రాసెస్ చేయడానికి ప్లగ్గిన్లను ఉపయోగించే ఏజెంట్లు ఎలా సృష్టించాలో, ఖర్చు క్లైమ్ ఇమెయిల్ తయారుచేసుకోవడం మరియు ఖర్చు డేటాను pai చార్ట్ ఉపయోగించి ఎలా విజువలైజ్ చేయాలో చూపిస్తుంది. ఏజెంట్లకు పనుల సందర్భాన్ని ఆధారంగా ఫంక్షన్లను డైనమిక్గా ఎంపిక చేసే సామర్థ్యం కలదు.

దశలు:
1. OCR ఏజెంట్ లోకల్ రసీదు చిత్రాన్ని ప్రాసెస్ చేసి ప్రయాణ ఖర్చు డేటాను తీయడం.
2. ఇమెయిల్ ఏజెంట్ ఖర్చు క్లెయిమ్ ఇమెయిల్ తయారు చేయడం.

### ప్రయాణ ఖర్చు పరిస్థితి యొక్క ఉదాహరణ:
మీరు మరో నగరంలో వ్యాపార సమావేశం కోసం ప్రయాణిస్తున్న ఉద్యోగి అని ఊహించండి. మీ కంపెనీ అన్ని తగిన ప్రయాణ సంబంధిత ఖర్చులను తిరిగి చెల్లించే విధానాన్ని కలిగి ఉంది. ఇక్కడ ప Potential ప్రయాణ ఖర్చుల వివరాలు:
- రవాణా:
మీ గృహ నగరంలో నుండి గమ్యమైన నగరానికి రౌండ్ ట్రిప్ కోసం విమాన టికెట్లు.
ఎయిర్‌పోర్ట్కు మరియు అక్కడి నుండి టాక్సీ లేదా రైడ్-హెయిలింగ్ సర్వీసులు.
గమ్య నగరంలో స్థానిక రవాణా (జనరల్ ట్రాన్సిట్, రెంటల్ కార్లు లేదా టాక్సీలు).

- వసతి:
సమావేశ ప్రాంగణానికి సమీపంలోని మిడ్-రేంజ్ బిజినెస్ హోటల్ లో మూడు రాత్రులు.

- భోజనాలు:
ప్రతిరోజూ బ్రేక్‌ఫాస్ట్, లంచ్ మరియు డిన్నర్ కోసం భోజన అనుమతి, కంపెనీ ప్రతిరోజూ విధానం ఆధారంగా.

- వివిధ ఖర్చులు:
ఎయిర్‌పోర్ట్ వద్ద పార్కింగ్ ఫీజులు.
హోటల్ లో ఇంటర్నెట్ యాక్సెస్ ఛార్జీలు.
టిప్స్ లేదా చిన్న సర్వీస్ ఛార్జీలు.

- డాక్యుమెంటేషన్:
మీరు అన్ని రసీదులు (ఫ్లైట్స్, టాక్సీలు, హోటల్, భోజనాలు, మొదలగు) మరియు పరిహారానికి పూర్తి చేసిన ఖర్చు నివేదిక సమర్పిస్తారు.


## అవసరమైన లైబ్రరీలను దిగుమతి చేయండి

నోట్‌బుక్ కోసం అవసరమైన లైబ్రరీలు మరియు మాడ్యూల్స్‌ను దిగుమతి చేయండి.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ఖర్చు మోడల్స్ నిర్వచించండి

వ్యక్తిగత ఖర్చుల కోసం ఒక Pydantic మోడల్ మరియు ఒక ExpenseFormatter క్లాస్‌ను సృష్టించండి, ఇది యూజర్ ప్రశ్నను నిర్మాణబద్ధమైన ఖర్చు డేటాగా మార్చుతుంది.

ప్రతి ఖర్చు ఈ ఫార్మాట్‌లో ప్రాతినిధ్యం వహిస్తుంది:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## పరికరాలను నిర్వచించడం - ఇమెయిల్ తయారు చేయడం

ఖర్చు క్లెయిమ్ సమర్పించేందుకు ఇమెయిల్ తయారు చేయడానికి ఒక పరికరం ఫంక్షనును సృష్టించండి.
- ఈ పరికరం Microsoft Agent Framework నుండి `@tool` డెకరేటర్ ఉపయోగిస్తుంది.
- ఇది ఖర్చుల మొత్తం మొత్తాన్ని లెక్కించుకుంటుంది మరియు వివరాలను ఒక ఇమెయిల్ బాడీగా ఫార్మాట్ చేస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులు తీసే సాధనం

రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను తీసే సాధన ఫంక్షన్‌ని సృష్టించండి.
- ఈ సాధనం Microsoft Agent Framework నుండి `@tool` డెకొరేటర్‌ను ఉపయోగిస్తుంది.
- ఇది రసీదు చిత్రాన్ని చదివి, దాన్ని base64గా కోడ్ చేసి, ఏజెంట్ విశ్లేషించడానికి డేటా URIని అందిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ఖర్చులను ప్రాసెస్ చేయడం

ఏజెంట్ల‌ను నిర్వచించి, వారిని `WorkflowBuilder` ఉపయోగించి ఒక వరుస వర్క్‌ఫ్లోలో లింక్ చేయండి.
- OCR ఏజెంట్ రసీదు చిత్రంలో నుండి గడచిన ఖర్చుల డేటాను `load_receipt_image` సాధనాన్ని ఉపయోగించి స结构ితంగా ఎగుమతి చేస్తుంది.
- ఈమెయిల్ ఏజెంట్ ఎగుమతి చేయబడిన డేటాను తీసుకుని `generate_expense_email` సాధనంతో ప్రొఫెషనల్ ఖర్చుల క్లెయిమ్ ఈమెయిల్‌ను రూపొందిస్తుంది.
- `WorkflowBuilder`లోని `add_edge`తో వరుస పైప్‌లైన్‌ను సృష్టించండి: OCR ఏజెంట్ → ఈమెయిల్ ఏజెంట్.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## ప్రధాన కార్యాచరణ

సీక్వెన్షియల్ వర్క్‌ఫ్లోని నిర్మించి, దానిని అమలు చేసి రసీదు చిత్రం ప్రాసెస్ చేసి ఖర్చుల క్లెయిమ్ ఇమెయిల్‌ను తయారు చేయండి.


> **గమనిక:** ఈ వర్క్‌ఫ్లో ప్రస్తుతం రసీదు చిత్రాన్ని base64 టెక్స్ట్‌గా పంపుతుంది, ఇది గిపిటి-4ో సహా చాలా చాట్ మోడళ్లచే చిత్రంగా పరిగణించబడదు.  
> ఇది మోడల్ కాంటెక్స్ట్ విండో కూడా మించవచ్చు. Azure AI Vision (లేదా మరొక OCR టూల్) తో OCR నిర్వహించడం మరియు తీసుకోబడిన టెక్స్ట్ను మాత్రమే పంపడం, లేదా చిత్రాన్ని `image_url` సందేశంగా పంపే విధంగా రీఫాక్టర్ చేయడం మంచిది.  
> మీరు కేవలం కాంటెక్స్ట్ లోపాలను నివారించాలనుకుంటే, తక్కువ పరిమాణం ఉన్న రసీదు చిత్రం లేదా ఎక్కువ కాంటెక్స్ట్ విండో ఉన్న మోడల్ వాడండి.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
